# Step 3.1: a feature, its parameters, and its versions

This notebook is the check of step 3.1. A feature is a Python function of the signal bars, the frame `ff.market.load` gives, written under `@ff.feature(...)`, which declares its parameters' defaults, its outputs and what each one is (a price, a distance in price, a ratio, a count, a flag, or other), and its warm-up, the bars before its values mean what they say. FEFAT keeps the function's text, decorator included, and hashes it with the schema of its parameters into the feature's content hash: the same text gives the same hash in any file and on any PC, and any change gives another. `ff.features.save` writes a feature as a version record that never changes, numbered from 1 per name, and `ff.features.load` reads one back and checks it.

The notebook defines a feature with one parameter, computes it on your last week of 5-minute bars, saves it, prints its version record, loads it back, and shows that a changed default and a changed character give other hashes. Pick the `.venv` kernel and run all cells. It reads the store in `FEFAT_DATA_DIR` and writes the feature's two versions, `v1.json` and `v2.json`, under `FEFAT_HOME/workspaces/default/features/stretch/`; running it again writes nothing new.

## A feature with a parameter

`stretch` is how far the close lies from the mean of the last `n` closes, in the bars' 1e-9 units of price, so its output is a `price_distance`; `n` is an integer that defaults to 20, and the warm-up is `n` bars.

Expected: the schema of its parameters, `n` an integer with the default 20, then its hash, 64 hexadecimal characters.

In [ ]:
import json
import os
from datetime import timedelta

import polars as pl
from IPython.display import display

import fefat as ff
from fefat.config import resolve_paths


@ff.feature(params={"n": 20}, warmup=lambda n: n, outputs={"stretch": "price_distance"})
def stretch(bars: pl.LazyFrame, n: int) -> pl.Expr:
    return pl.col("close") - pl.col("close").rolling_mean(n)


print(json.dumps(stretch.parameter_schema, indent=2))
print("hash:", stretch.hash)

## Computed on your bars

The product: MES when your store holds its 1-minute bars, else the product with the most contracts, and its 5-minute bars of the week to its last trade date. `stretch(n=20)` binds the parameter, checked as the schema says, and `evaluate` gives a frame of `ts_close`, `stretch`, and `warmup`, true on the first 20 rows.

Expected: one line with the product, its bars, those in the warm-up, and the values, then the last five rows, the stretch in points.

In [ ]:
paths = resolve_paths(os.environ)
listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    raise ValueError(
        f"Expected 1-minute bars of at least one product in {paths.data_dir}, got none: "
        "import your files first (docs/data-guide.md)."
    )
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, PROVIDER, LAST = chosen["root"], chosen["provider"], chosen["last_trade_date"]
bars = ff.market.load(ROOT, "5m", LAST - timedelta(days=7), LAST, provider=PROVIDER)
values = stretch(n=20).evaluate(bars).collect()
print(
    f"{ROOT} from {PROVIDER} to {LAST}: {values.height} bars, {values['warmup'].sum()} in the "
    f"warm-up, {values['stretch'].drop_nulls().len()} values"
)
display(
    values.tail(5).select(
        pl.from_epoch("ts_close", time_unit="ns").alias("close_utc"),
        (pl.col("stretch") / 1e9).alias("stretch_points"),
        "warmup",
    )
)

## Saved as version 1

`ff.features.save(stretch)` writes the feature's version record and returns it; a feature whose hash a saved version already has returns that version and writes nothing.

Expected: the file's path, then the record as the file holds it: its `name`, its `version` 1, its `hash`, its `code` (the decorator and the function as written above), its `parameter_schema`, its `inputs` (none beside the bars), its `outputs`, its `warmup`, and the time it was saved.

In [ ]:
saved = ff.features.save(stretch)
record_file = paths.features_dir / saved.name / f"v{saved.version}.json"
print(record_file)
print(record_file.read_text(encoding="utf-8"))

## Loaded back

`ff.features.load` reads the record, works its hash out again from its code and schema, and holds its outputs, inputs, and warm-up to what its code's decorator writes, refusing a file changed after it was written.

Expected: `version 1, identical hash: True`.

In [ ]:
loaded = ff.features.load("stretch", saved.version)
print(f"version {loaded.version}, identical hash: {loaded.hash == saved.hash == stretch.hash}")

## A changed default and a changed character

The same feature with `n` defaulting to 30 has another schema, so another hash, and saving it writes version 2. With one character of its code changed, `+` for `-`, the hash changes again; that one is not saved.

Expected: the three hashes, version 2 for the changed default, and `all differ: True`.

In [ ]:
@ff.feature(params={"n": 30}, warmup=lambda n: n, outputs={"stretch": "price_distance"})
def stretch(bars: pl.LazyFrame, n: int) -> pl.Expr:
    return pl.col("close") - pl.col("close").rolling_mean(n)


changed_default = stretch
changed_default_version = ff.features.save(changed_default)


@ff.feature(params={"n": 20}, warmup=lambda n: n, outputs={"stretch": "price_distance"})
def stretch(bars: pl.LazyFrame, n: int) -> pl.Expr:
    return pl.col("close") + pl.col("close").rolling_mean(n)


changed_character = stretch
print("first:          ", saved.hash)
print("default 30:     ", changed_default.hash, "version", changed_default_version.version)
print("one character:  ", changed_character.hash)
print("all differ:", len({saved.hash, changed_default.hash, changed_character.hash}) == 3)